In [1]:
import pandas as pd
import geopandas as gpd
import re
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
neighbour_profiles = pd.read_excel('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/neighbourhood-profiles-2021-158-model.xlsx') # demographic data 
child_centres = gpd.read_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/EarlyON Child and Family Centres Locations - geometry - 4326.geojson') # childcare centre location data
improvement_areas = gpd.read_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Neighbourhood Improvement Areas - 4326.geojson') # neighbourhood improvement area location data
neighbourhoods = gpd.read_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Neighbourhoods - 4326.geojson') # neighbourhood location data

In [3]:
neighbour_profiles
transpose = neighbour_profiles.set_index(neighbour_profiles.columns[0]).T # transpose data
transpose = transpose.reset_index().rename(columns={'index': 'Neighbourhood'}) # reset the index to make neighbourhood index
transpose

Neighbourhood Name,Neighbourhood,Neighbourhood Number,TSNS 2020 Designation,Total - Age groups of the population - 25% sample data,0 to 14 years,0 to 4 years,5 to 9 years,10 to 14 years,15 to 64 years,15 to 19 years,...,Between 9 a.m. and 11:59 a.m.,Between 12 p.m. and 4:59 a.m.,Total - Eligibility for instruction in the minority official language for the population in private households born in 2003 or later - 25% sample data,Children eligible for instruction in the minority official language,Children not eligible for instruction in the minority official language,"Total - Eligibility and instruction in the minority official language, for the population in private households born between 2003 and 2015 (inclusive) - 25% sample data",Children eligible for instruction in the minority official language,Eligible children who have been instructed in the minority official language at the primary or secondary level in Canada,Eligible children who have not been instructed in the minority official language at the primary or secondary level in Canada,Children not eligible for instruction in the minority official language
0,West Humber-Clairville,1,Not an NIA or Emerging Neighbourhood,33300,4295,1460,1345,1485,23640,1860,...,1665,2935,5430,410,5020,3875,335,255,75,3540
1,Mount Olive-Silverstone-Jamestown,2,Neighbourhood Improvement Area,31345,5690,1650,1860,2175,21490,2280,...,1145,2965,7285,510,6780,5540,395,245,145,5145
2,Thistletown-Beaumond Heights,3,Neighbourhood Improvement Area,9850,1495,505,540,455,6615,570,...,395,635,1860,180,1685,1325,120,75,45,1205
3,Rexdale-Kipling,4,Not an NIA or Emerging Neighbourhood,10375,1575,505,615,455,6950,515,...,425,775,1910,135,1770,1370,90,75,25,1275
4,Elms-Old Rexdale,5,Neighbourhood Improvement Area,9355,1610,440,480,685,6355,635,...,355,675,2015,95,1920,1520,70,60,10,1445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,Yonge-Bay Corridor,170,Not an NIA or Emerging Neighbourhood,12645,970,500,270,200,10820,340,...,695,330,1075,50,1020,555,30,10,20,520
154,Junction-Wallace Emerson,171,Not an NIA or Emerging Neighbourhood,23180,3075,1135,1020,925,17200,750,...,1185,1105,3580,410,3170,2375,305,190,115,2075
155,Dovercourt Village,172,Not an NIA or Emerging Neighbourhood,12380,1365,445,430,490,9040,460,...,775,570,1665,175,1490,1190,130,95,35,1060
156,North Toronto,173,Not an NIA or Emerging Neighbourhood,15885,1315,535,390,390,12780,465,...,970,550,1620,145,1470,1050,95,65,30,955


In [4]:
transpose.to_csv('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/neighbourhood_profiles_clean.csv', index=False)

In [5]:
transpose[['Neighbourhood', '    Couple-family households', '      With children']]

Neighbourhood Name,Neighbourhood,Couple-family households,With children,With children,With children
0,West Humber-Clairville,4515,2945,3930,215
1,Mount Olive-Silverstone-Jamestown,4050,2950,3605,275
2,Thistletown-Beaumond Heights,1380,905,1105,100
3,Rexdale-Kipling,1600,995,1115,110
4,Elms-Old Rexdale,1210,740,895,100
...,...,...,...,...,...
153,Yonge-Bay Corridor,2095,660,660,50
154,Junction-Wallace Emerson,4360,2200,2185,340
155,Dovercourt Village,2250,1070,1120,145
156,North Toronto,3080,990,880,135


In [6]:
list(transpose['Neighbourhood'])

['West Humber-Clairville',
 'Mount Olive-Silverstone-Jamestown',
 'Thistletown-Beaumond Heights',
 'Rexdale-Kipling',
 'Elms-Old Rexdale',
 'Kingsview Village-The Westway',
 'Willowridge-Martingrove-Richview',
 'Humber Heights-Westmount',
 'Edenbridge-Humber Valley',
 'Princess-Rosethorn',
 'Eringate-Centennial-West Deane',
 'Markland Wood',
 'Etobicoke West Mall',
 'Kingsway South',
 'Stonegate-Queensway',
 'New Toronto',
 'Long Branch',
 'Alderwood',
 'Humber Summit',
 'Humbermede',
 'Pelmo Park-Humberlea',
 'Black Creek',
 'Glenfield-Jane Heights',
 'York University Heights',
 'Rustic',
 'Maple Leaf',
 'Brookhaven-Amesbury',
 'Yorkdale-Glen Park',
 'Englemount-Lawrence',
 'Clanton Park',
 'Bathurst Manor',
 'Westminster-Branson',
 'Newtonbrook West',
 'Willowdale West',
 'Lansing-Westgate',
 'Bedford Park-Nortown',
 'St.Andrew-Windfields',
 'Bridle Path-Sunnybrook-York Mills',
 'Banbury-Don Mills',
 'Victoria Village',
 'Flemingdon Park',
 'Pleasant View',
 'Don Valley Village',
 'H

In [7]:
child_centres["dropinHours"].unique()

array(['Wednesday: 9:00 a.m. - 11:30 a.m.  ', None,
       'Monday: 9:00 a.m. - noon  ; 1:00 p.m. - 3:30 p.m.  ; 1:30 p.m. - 2:30 p.m.   | Tuesday: 9:00 a.m. - noon  ; 10:00 a.m. - 11:00 a.m.  ; 1:00 p.m. - 3:30 p.m.   | Wednesday: 9:00 a.m. - noon  ; 10:00 a.m. - 11:00 a.m.  ; 1:00 p.m. - 3:30 p.m.   | Thursday: 9:00 a.m. - noon  ; 1:00 p.m. - 3:30 p.m.  ; 4:30 p.m. - 7:00 p.m.   | Friday: 9:00 a.m. - noon  ; 1:00 p.m. - 3:30 p.m.   | Saturday: 9:30 a.m. - noon  ',
       'Monday: 10:00 a.m. - noon  ; 3:30 p.m. - 6:00 p.m.   | Tuesday: 2:00 p.m. - 4:00 p.m.   | Wednesday: 10:00 a.m. - noon  ; 2:00 p.m. - 4:00 p.m.   | Thursday: 2:00 p.m. - 4:00 p.m.   | Friday: 10:00 a.m. - noon   | Saturday: 10:00 a.m. - noon  ',
       'Monday: 9:00 a.m. - 11:30 a.m.  ; 1:00 p.m. - 3:30 p.m.   | Tuesday: 1:00 p.m. - 4:00 p.m.   | Wednesday: 9:00 a.m. - 11:30 a.m.   | Thursday: 1:00 p.m. - 3:30 p.m.   | Friday: 9:00 a.m. - 11:30 a.m.  ; 1:00 p.m. - 3:30 p.m.  ',
       'Wednesday: 1:00 p.m. - 3:30 p.

In [8]:
def categorize_hours(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return "Other"
    
    t = text.lower()
    
    # Detect days
    days = re.findall(r"(monday|tuesday|wednesday|thursday|friday|saturday|sunday)", t)
    unique_days = set(days)
    
    # Detect if weekend present
    weekend = bool({"saturday", "sunday"} & unique_days)
    
    # Detect time ranges
    evening = bool(re.search(r"(5|6|7|8|9):\d{0,2}\s*p\.?m\.?", t))
    morning = bool(re.search(r"(7|8|9|10|11):\d{0,2}\s*a\.?m\.?", t))
    afternoon = bool(re.search(r"(12|1|2|3|4):\d{0,2}\s*p\.?m\.?", t))
    
    # Apply logic
    if weekend:
        return "Weekend available"
    elif evening:
        return "Evening available"
    elif morning and afternoon:
        return "Full day"
    elif morning:
        return "Morning only"
    elif afternoon:
        return "Afternoon only"
    else:
        return "Other"

child_centres["hourCategory"] = child_centres["dropinHours"].apply(categorize_hours)

child_centres["hourCategory"]

0           Morning only
1                  Other
2      Weekend available
3      Weekend available
4               Full day
             ...        
225    Weekend available
226    Weekend available
227                Other
228             Full day
229    Evening available
Name: hourCategory, Length: 230, dtype: object

In [9]:
child_centres['address']

0            101 Spruce St
1           1033 King St W
2      2555 Eglinton Ave E
3         2700 Dufferin St
4             38 Regent St
              ...         
225     160 Eglinton Ave E
226      30 Sheppard Ave E
227       1785 Finch Ave W
228      25 Yorkwoods Gate
229       3090 Kingston Rd
Name: address, Length: 230, dtype: object

In [10]:
child_centres.to_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/child_centres_clean.geojson', driver="GeoJSON")

In [11]:
# In 2022, Toronto designated 33 neighbourhoods as vulnerable areas compared to 14 in 2014.

(33 - 14) / 14 * 100

135.71428571428572

In [12]:
list(improvement_areas.columns)

['_id',
 'AREA_ID',
 'DATE_EFFECTIVE',
 'DATE_EXPIRY',
 'AREA_ATTR_ID',
 'AREA_TYPE_ID',
 'PARENT_AREA_ID',
 'AREA_TYPE',
 'AREA_CLASS_ID',
 'AREA_CLASS',
 'AREA_SHORT_CODE',
 'AREA_LONG_CODE',
 'AREA_NAME',
 'AREA_DESC',
 'FEATURE_CODE',
 'FEATURE_CODE_DESC',
 'TRANS_ID_CREATE',
 'TRANS_ID_EXPIRE',
 'OBJECTID',
 'geometry']

In [13]:
list(neighbourhoods.columns)

['_id',
 'AREA_ID',
 'AREA_ATTR_ID',
 'PARENT_AREA_ID',
 'AREA_SHORT_CODE',
 'AREA_LONG_CODE',
 'AREA_NAME',
 'AREA_DESC',
 'CLASSIFICATION',
 'CLASSIFICATION_CODE',
 'OBJECTID',
 'geometry']

In [14]:
matching_values = transpose['Neighbourhood'].isin(neighbourhoods['AREA_NAME'])

In [15]:
missing = neighbourhoods[~neighbourhoods['AREA_NAME'].isin(transpose['Neighbourhood'])]
print(missing['AREA_NAME'])

60                      Yonge-St.Clair
81                 North St.James Town
84     Cabbagetown-South St.James Town
93                   East End-Danforth
94                       Taylor-Massey
96                  Danforth East York
130                  O'Connor-Parkview
Name: AREA_NAME, dtype: object


In [16]:
matching_values2 = transpose['Neighbourhood'].isin(improvement_areas['AREA_NAME'])
count_true = matching_values2.sum()
print(count_true)

32


In [17]:
len(improvement_areas)

33

In [18]:
missing_from_transpose = improvement_areas[~improvement_areas['AREA_NAME'].isin(transpose['Neighbourhood'])]
print(missing_from_transpose)


    _id    AREA_ID      DATE_EFFECTIVE          DATE_EXPIRY  AREA_ATTR_ID  \
18   19  2502272.0 2022-04-11 21:27:23  3000/01/01 05:00:00    26022787.0   

    AREA_TYPE_ID PARENT_AREA_ID AREA_TYPE AREA_CLASS_ID AREA_CLASS  \
18         602.0           None      CNBH          None       None   

   AREA_SHORT_CODE AREA_LONG_CODE      AREA_NAME           AREA_DESC  \
18             061            061  Taylor-Massey  Taylor-Massey (61)   

   FEATURE_CODE FEATURE_CODE_DESC  TRANS_ID_CREATE  TRANS_ID_EXPIRE  OBJECTID  \
18         None              None         344709.0             -1.0  17826241   

                                             geometry  
18  MULTIPOLYGON (((-79.29348 43.70302, -79.29314 ...  


## **Preprocessing**

In [19]:
neighbourhood_metrics = pd.DataFrame()

neighbourhood_metrics['neighbourhood_name'] = transpose['Neighbourhood']
neighbourhood_metrics['children_0_4'] = transpose['    0 to 4 years'] # childcare centers are for infants to preschool
neighbourhood_metrics['median_age_population'] = transpose['Median age of the population']
neighbourhood_metrics['median_after_tax_income_2020_over_15'] = transpose[ '    Median after-tax income in 2020 among recipients ($)']
neighbourhood_metrics['after_tax_bottom_half_distribution'] = transpose['  In bottom half of the distribution']
neighbourhood_metrics[ '    In bottom decile'] = transpose['    In bottom decile']
neighbourhood_metrics[ '    In second decile'] = transpose['    In second decile']
neighbourhood_metrics[ '    In third decile'] = transpose['    In third decile']
neighbourhood_metrics[ '    In fourth decile'] = transpose['    In fourth decile']
neighbourhood_metrics[ '    In fifth decile'] = transpose['    In fifth decile']
neighbourhood_metrics[ '    In sixth decile'] = transpose['    In sixth decile']
neighbourhood_metrics[ '    In seventh decile'] = transpose['    In seventh decile']
neighbourhood_metrics[ '    In eighth decile'] = transpose['    In eighth decile']
neighbourhood_metrics[ '    In ninth decile'] = transpose['    In ninth decile']
neighbourhood_metrics[ '    In top decile'] = transpose['    In top decile']
neighbourhood_metrics['inequality_index'] = transpose['  Gini index on adjusted household after-tax income']

with_children_cols = transpose.filter(like='With children')
transpose['count_of_children'] = with_children_cols.sum(axis=1) # row wise sum
neighbourhood_metrics['count_of_children'] = transpose['count_of_children']

neighbourhood_metrics['single_moms'] = transpose[ '    in which the parent is a woman+']

nias = improvement_areas['AREA_NAME'].unique() 
neighbourhood_metrics['is_nia'] = neighbourhood_metrics['neighbourhood_name'].isin(nias)


In [20]:
neighbourhoods = neighbourhoods.to_crs(child_centres.crs)

# assigning child care centres to neighbourhoods
centres_per_neighbourhood = gpd.sjoin(child_centres, neighbourhoods, how='left', predicate='within') # spatial join

childcare_counts = centres_per_neighbourhood.groupby('AREA_NAME').size().reset_index(name='childcare_centres_count') 

neighbourhood_metrics = neighbourhood_metrics.merge(
    childcare_counts, 
    left_on='neighbourhood_name', 
    right_on='AREA_NAME', 
    how='left'
)
neighbourhood_metrics['childcare_centres_count'] = neighbourhood_metrics['childcare_centres_count'].fillna(0)

In [43]:
neighbourhood_metrics

,AREA_NAME,children_0_4,median_age_population,median_after_tax_income_2020_over_15,after_tax_bottom_half_distribution,In bottom decile,In second decile,In third decile,In fourth decile,In fifth decile,...,In eighth decile,In ninth decile,In top decile,inequality_index,count_of_children,single_moms,is_nia,AREA_NAME,childcare_centres_count,centres_per_1000_children
0,West Humber-Clairville,1460,38,31600,17320,3430,3460,3270,3585,3580,...,3485,3300,1610,0.2,7090,1695,False,West Humber-Clairville,4.0,2.739726
1,Mount Olive-Silverstone-Jamestown,1650,36,28400,21365,4465,4885,4645,3915,3460,...,2155,1240,455,0.2,6830,2060,True,Mount Olive-Silverstone-Jamestown,5.0,3.030303
2,Thistletown-Beaumond Heights,505,39.2,30600,5790,1215,1030,1160,1240,1145,...,845,735,595,0.3,2110,540,True,NaN,0.0,0.000000
3,Rexdale-Kipling,505,42,31800,5795,1390,1155,1225,1105,925,...,1010,1035,460,0.3,2220,630,False,NaN,0.0,0.000000
4,Elms-Old Rexdale,440,38.4,32400,5150,1075,1045,1250,1030,750,...,740,740,405,0.3,1735,740,True,Elms-Old Rexdale,3.0,6.818182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,Yonge-Bay Corridor,500,31,40400,6880,2985,1420,1030,765,670,...,965,1195,1920,0.4,1370,335,False,NaN,0.0,0.000000
154,Junction-Wallace Emerson,1135,37.6,37200,11315,2595,2250,2080,2205,2185,...,2480,2365,2775,0.3,4725,940,False,Junction-Wallace Emerson,2.0,1.762115
155,Dovercourt Village,445,39.2,34800,6000,1435,1360,1300,750,1155,...,1125,1225,1970,0.3,2335,475,False,Dovercourt Village,3.0,6.741573
156,North Toronto,535,35.2,41200,8480,2630,1595,1555,1240,1465,...,1525,1340,1430,0.3,2005,570,False,North Toronto,1.0,1.869159


In [47]:
neighbourhood_metrics.columns

Index(['AREA_NAME', 'children_0_4', 'median_age_population',
       'median_after_tax_income_2020_over_15',
       'after_tax_bottom_half_distribution', '    In bottom decile',
       '    In second decile', '    In third decile', '    In fourth decile',
       '    In fifth decile', '    In sixth decile', '    In seventh decile',
       '    In eighth decile', '    In ninth decile', '    In top decile',
       'inequality_index', 'count_of_children', 'single_moms', 'is_nia',
       'AREA_NAME', 'childcare_centres_count', 'centres_per_1000_children'],
      dtype='object')

In [49]:
neighbourhood_metrics[['AREA_NAME','    In third decile']]

,AREA_NAME,AREA_NAME,In third decile
0,West Humber-Clairville,West Humber-Clairville,3270
1,Mount Olive-Silverstone-Jamestown,Mount Olive-Silverstone-Jamestown,4645
2,Thistletown-Beaumond Heights,NaN,1160
3,Rexdale-Kipling,NaN,1225
4,Elms-Old Rexdale,Elms-Old Rexdale,1250
...,...,...,...
153,Yonge-Bay Corridor,NaN,1030
154,Junction-Wallace Emerson,Junction-Wallace Emerson,2080
155,Dovercourt Village,Dovercourt Village,1300
156,North Toronto,North Toronto,1555


In [22]:
# looked at distribution of count per children and 100 is not representative

In [23]:
neighbourhood_metrics['centres_per_1000_children'] = (
    neighbourhood_metrics['childcare_centres_count'] / 
    neighbourhood_metrics['children_0_4'] * 1000
).replace([np.inf, -np.inf], 0).fillna(0) # calculating the density of childcare centres per 1000 children


/var/folders/3n/b26hkpfd71x2jvpstpknt4ch0000gn/T/ipykernel_40989/2962788464.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).replace([np.inf, -np.inf], 0).fillna(0) # calculating the density of childcare centres per 1000 children


In [24]:
childcare_centres_clean = pd.DataFrame({
    'centre_id': child_centres.index,
    'address': child_centres['address'],   
    'geometry': child_centres['geometry'],
    'operating_hours': child_centres['hourCategory'],
    'Area Name': centres_per_neighbourhood['AREA_NAME']
})

In [33]:
childcare_centres_clean.columns

Index(['centre_id', 'address', 'geometry', 'operating_hours', 'Area Name'], dtype='object')

In [26]:
# looking at inequality in NIAs versus other neighbourhoods
nia_stats = neighbourhood_metrics[neighbourhood_metrics['is_nia'] == True]
non_nia_stats = neighbourhood_metrics[neighbourhood_metrics['is_nia'] == False]

In [27]:
print("=== EQUITY GAP ANALYSIS ===")
print(f"NIA neighbourhoods ({len(nia_stats)}): {nia_stats['centres_per_1000_children'].mean():.1f} centres per 100 children")
print(f"Non-NIA neighbourhoods ({len(non_nia_stats)}): {non_nia_stats['centres_per_1000_children'].mean():.1f} centres per 100 children")
print(f"Gap: {non_nia_stats['centres_per_1000_children'].mean() - nia_stats['centres_per_1000_children'].mean():.1f} centres per 100 children")

# Find childcare deserts (e.g., fewer than 1 centre per 100 children)
childcare_deserts = neighbourhood_metrics[neighbourhood_metrics['centres_per_1000_children'] < 1]
print(f"\nChildcare deserts (fewer than 1 centre per 100 children): {len(childcare_deserts)} neighbourhoods")
print(f"Of these, {childcare_deserts['is_nia'].sum()} are NIA neighbourhoods")

=== EQUITY GAP ANALYSIS ===
NIA neighbourhoods (32): 2.8 centres per 100 children
Non-NIA neighbourhoods (126): 1.6 centres per 100 children
Gap: -1.2 centres per 100 children

Childcare deserts (fewer than 1 centre per 100 children): 60 neighbourhoods
Of these, 3 are NIA neighbourhoods


In [28]:
# Rename the column to match exactly
neighbourhood_metrics = neighbourhood_metrics.rename(columns={'neighbourhood_name': 'AREA_NAME'})

# Then export
neighbourhood_metrics.to_csv('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/toronto_childcare_metrics.csv', index=False)

In [29]:
neighbourhood_metrics.to_csv('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/toronto_childcare_metrics.csv', index=False)

childcare_centres_clean.to_csv('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/toronto_childcare_centres.csv', index=False)

neighbourhoods.to_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/toronto_neighbourhoods_clean.geojson', driver='GeoJSON')
improvement_areas.to_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/nia_neighbourhoods_clean.geojson', driver='GeoJSON')
child_centres.to_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/childcare_locations_clean.geojson', driver='GeoJSON')

In [50]:
df = pd.read_csv('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/toronto_childcare_metrics.csv')
df2 = pd.read_csv('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/toronto_childcare_centres.csv')
df3 = gpd.read_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/toronto_neighbourhoods_clean.geojson')
df4 = gpd.read_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/nia_neighbourhoods_clean.geojson')
df5 = gpd.read_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/childcare_locations_clean.geojson')

In [52]:
print(df.columns)
print(df2.columns)
print(df3.columns)
print(df4.columns)
print(df5.columns)

Index(['AREA_NAME', 'children_0_4', 'median_age_population',
       'median_after_tax_income_2020_over_15',
       'after_tax_bottom_half_distribution', '    In bottom decile',
       '    In second decile', '    In third decile', '    In fourth decile',
       '    In fifth decile', '    In sixth decile', '    In seventh decile',
       '    In eighth decile', '    In ninth decile', '    In top decile',
       'inequality_index', 'count_of_children', 'single_moms', 'is_nia',
       'AREA_NAME.1', 'childcare_centres_count', 'centres_per_1000_children'],
      dtype='object')
Index(['centre_id', 'address', 'geometry', 'operating_hours', 'Area Name'], dtype='object')
Index(['_id', 'AREA_ID', 'AREA_ATTR_ID', 'PARENT_AREA_ID', 'AREA_SHORT_CODE',
       'AREA_LONG_CODE', 'AREA_NAME', 'AREA_DESC', 'CLASSIFICATION',
       'CLASSIFICATION_CODE', 'OBJECTID', 'geometry'],
      dtype='object')
Index(['_id', 'AREA_ID', 'DATE_EFFECTIVE', 'DATE_EXPIRY', 'AREA_ATTR_ID',
       'AREA_TYPE_ID', 'PARE

In [53]:
# 1. Fix the main metrics table (df)
# Drop the redundant column first
df = df.drop(columns=['AREA_NAME.1']) 
df = df.rename(columns={'AREA_NAME': 'Neighbourhood_Name'})

# 2. Fix the childcare centres table (df2)
df2 = df2.rename(columns={'Area Name': 'Neighbourhood_Name'})

# 3. Fix the main neighbourhood polygons table (df3)
df3 = df3.rename(columns={'AREA_NAME': 'Neighbourhood_Name'})

# 4. Fix the NIA neighbourhood table (df4)
df4 = df4.rename(columns={'AREA_NAME': 'Neighbourhood_Name'})

print("Renaming Complete. All join keys are now 'Neighbourhood_Name'.")

Renaming Complete. All join keys are now 'Neighbourhood_Name'.


In [54]:
# Master GeoDataFrame: Start with Polygons (df3) and Left Join the Metrics (df)
master_df = df3.merge(df, on='Neighbourhood_Name', how='left')

In [55]:
master_df.to_file('/Users/lucasben/Documents/mba-business-analytics/Data Viz Data Folder/Tableau Data/master_df.geojson', driver='GeoJSON')

In [56]:
master_df.columns

Index(['_id', 'AREA_ID', 'AREA_ATTR_ID', 'PARENT_AREA_ID', 'AREA_SHORT_CODE',
       'AREA_LONG_CODE', 'Neighbourhood_Name', 'AREA_DESC', 'CLASSIFICATION',
       'CLASSIFICATION_CODE', 'OBJECTID', 'geometry', 'children_0_4',
       'median_age_population', 'median_after_tax_income_2020_over_15',
       'after_tax_bottom_half_distribution', '    In bottom decile',
       '    In second decile', '    In third decile', '    In fourth decile',
       '    In fifth decile', '    In sixth decile', '    In seventh decile',
       '    In eighth decile', '    In ninth decile', '    In top decile',
       'inequality_index', 'count_of_children', 'single_moms', 'is_nia',
       'childcare_centres_count', 'centres_per_1000_children'],
      dtype='object')

In [65]:
master_df['after_tax_bottom_half_distribution']

0      10335.0
1       8480.0
2       6000.0
3      11315.0
4       6880.0
        ...   
153    17320.0
154    15765.0
155     4975.0
156    11030.0
157     7500.0
Name: after_tax_bottom_half_distribution, Length: 158, dtype: float64